# 00 - Use a volumetric mask to generate a set of fibers from a structural connectome
- This set of fibers will be used to perform fiber filtering
- ConnectomeSeed takes several args:
    - connectome_path - path to a structural connectome file (ends in .trk/.tck/.trx/.mat)
    - out - path where your fiber atlas will be saved to
    - seed_mask - this is the nifti mask which will generate the fibers
    - exclude_mask - this is a list of masks which will be excluded if a fiber hits them
    - intersect_mask - this is a list of masks which fibers must pass through

- notes
    - Masks must be in the same space as the structural connectome. 
    - Fibers are generated in the resolution of the input mask. Masks are NOT resampled to the connectome. 


In [ ]:
from calvin_utils.neuroimaging_utils.tract_utils.connectome_seed import ConnectomeSeed

seeded = ConnectomeSeed(
    connectome_path="/Volumes/HowExp/resources/connectome_dMRI/i74(Full)/data.trk",
    seed_mask="/Users/cu135/hires_backdrops/suit/atl-Anatom_space-MNI_dseg_coverage.nii.gz",
    intersect_mask=["/Volumes/HowExp/resources/atlases/mni_space/MNI_structures/subcortex/subcortex_mask_2mm.nii.gz"],
    exclude_mask=[],
    out="/Volumes/OneTouch/01p_Schmahmann_SCA_Atrophy/connectomic/um1_fibers",
    max_fibers=200000
)


In [ ]:
from pathlib import Path
import numpy as np
import nibabel as nib
from nibabel.streamlines import LazyTractogram, TrkFile

files = sorted(
    p for p in Path(
        "/Volumes/HowExp/resources/connectome_dMRI/HCP1065_fiber_atlas/individual_fiber_atlas"
    ).rglob("*.trk.gz")
    if not p.name.startswith("._")
)
print(f"Found {len(files)} tract files")

header = nib.streamlines.load(str(files[0]), lazy_load=True).header.copy()

def streamlines():
    for file in files:
        trk = nib.streamlines.load(str(file), lazy_load=True)
        assert np.allclose(trk.header["voxel_to_rasmm"],
                           header["voxel_to_rasmm"])
        yield from trk.tractogram

merged = LazyTractogram.from_data_func(streamlines)
merged.affine_to_rasmm = np.eye(4)

nib.streamlines.save(
    TrkFile(merged, header=header),
    "/Volumes/HowExp/resources/connectome_dMRI/HCP1065_fiber_atlas/HCP1065_fiber_atlas.trk"
)